# Merchant Registration Batch

`merchant_info`의 상가 정보 CSV에서 `상호명`을 읽고, SSAFY 금융 API `inquireMerchantList`로 기존 가맹점을 조회한 뒤 `merchantName` 중복을 제거해 `createMerchant`를 호출합니다. 등록 중에는 `tqdm` 진행바로 현황을 확인할 수 있습니다.

- 고정 `categoryId`: `CG-9ca85f66311a23d`
- 기존 SSAFY 가맹점과 `merchantName`이 같으면 등록 대상에서 제외
- 성공 결과 CSV: `merchantId`, `merchantName`
- 실패 결과 CSV: `merchantName`, `responseCode`, `responseMessage`, `errorType`


In [1]:
%pip install python-dotenv tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import importlib
import os
import sys
from datetime import datetime
from pathlib import Path
from pprint import pprint

import httpx
from dotenv import load_dotenv
from tqdm.auto import tqdm

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / 'lab').exists() and (path / 'pyproject.toml').exists()
    ),
    current,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

from lab.finapi import merchant_registration

merchant_registration = importlib.reload(merchant_registration)

CREATE_MERCHANT_URL = merchant_registration.CREATE_MERCHANT_URL
DEFAULT_CATEGORY_ID = merchant_registration.DEFAULT_CATEGORY_ID
discover_merchant_csv_paths = merchant_registration.discover_merchant_csv_paths
exclude_existing_merchant_names = merchant_registration.exclude_existing_merchant_names
fetch_registered_merchant_names = merchant_registration.fetch_registered_merchant_names
load_merchant_names = merchant_registration.load_merchant_names
register_merchants = merchant_registration.register_merchants
write_failure_rows = merchant_registration.write_failure_rows
write_success_rows = merchant_registration.write_success_rows

INPUT_PATH = PROJECT_ROOT / 'lab' / 'merchant_info'
OUTPUT_DIR = PROJECT_ROOT / 'lab' / 'finapi' / 'outputs'
API_KEY = os.getenv('SSAFY_FIN_API_KEY', '')
INSTITUTION_CODE = os.getenv('SSAFY_FIN_INSTITUTION_CODE', '00100')
FINTECH_APP_NO = os.getenv('SSAFY_FIN_FINTECH_APP_NO', '001')
CATEGORY_ID = DEFAULT_CATEGORY_ID
BASE_URL = CREATE_MERCHANT_URL
MAX_MERCHANTS = None

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'INPUT_PATH: {INPUT_PATH}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'CATEGORY_ID: {CATEGORY_ID}')
print('API_KEY configured:', bool(API_KEY))


PROJECT_ROOT: C:\Users\SSAFY\more\more-ai\AI
INPUT_PATH: C:\Users\SSAFY\more\more-ai\AI\lab\merchant_info
OUTPUT_DIR: C:\Users\SSAFY\more\more-ai\AI\lab\finapi\outputs
CATEGORY_ID: CG-9ca85f66311a23d
API_KEY configured: True


c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
target_region_markers = ("_대구_", "_경북_")
csv_paths = [
    path
    for path in discover_merchant_csv_paths(INPUT_PATH)
    if any(marker in path.name for marker in target_region_markers)
]
merchant_names = load_merchant_names(csv_paths)

if MAX_MERCHANTS is not None:
    merchant_names = merchant_names[:MAX_MERCHANTS]

print(f'CSV files: {len(csv_paths)}')
print(f'Merchant names from CSV: {len(merchant_names)}')
print('Sample merchant names:')
pprint(merchant_names[:10])


CSV files: 2
Merchant names from CSV: 206098
Sample merchant names:
['한백종합기술공사',
 '펫매니저석적점',
 '달빛정원스튜디오',
 '디자인선',
 '하나공인중개사무소',
 '경북고용성장지원센터',
 '런휘트니스',
 '대림부동산컨설팅',
 '정민에스테틱',
 '걸앤맨']


In [4]:
if not API_KEY:
    raise ValueError('SSAFY_FIN_API_KEY 환경 변수가 비어 있습니다.')

with httpx.Client() as client:
    registered_merchant_names = fetch_registered_merchant_names(
        api_key=API_KEY,
        client=client,
        institution_code=INSTITUTION_CODE,
        fintech_app_no=FINTECH_APP_NO,
    )
    pending_merchant_names, skipped_merchant_names = exclude_existing_merchant_names(
        merchant_names=merchant_names,
        existing_merchant_names=registered_merchant_names,
    )

    print(f'Existing SSAFY merchant names: {len(registered_merchant_names)}')
    print(f'Skipped duplicates: {len(skipped_merchant_names)}')
    print(f'Pending registrations: {len(pending_merchant_names)}')

    run_at = datetime.now().strftime('%Y%m%d_%H%M%S')
    success_csv = OUTPUT_DIR / f'registered_merchants_{run_at}.csv'
    failure_csv = OUTPUT_DIR / f'merchant_registration_failures_{run_at}.csv'

    with tqdm(total=len(pending_merchant_names), desc="Registering merchants", unit="merchant") as progress_bar:
        def update_registration_progress(completed: int, total: int, merchant_name: str) -> None:
            progress_bar.total = total
            progress_bar.update(completed - progress_bar.n)
            progress_bar.set_postfix_str(merchant_name[:20], refresh=True)

        success_rows, failure_rows = register_merchants(
            merchant_names=pending_merchant_names,
            api_key=API_KEY,
            client=client,
            category_id=CATEGORY_ID,
            base_url=BASE_URL,
            institution_code=INSTITUTION_CODE,
            fintech_app_no=FINTECH_APP_NO,
            progress_callback=update_registration_progress,
        )

write_success_rows(success_csv, success_rows)
write_failure_rows(failure_csv, failure_rows)

print(f'Success rows: {len(success_rows)} -> {success_csv}')
print(f'Failure rows: {len(failure_rows)} -> {failure_csv}')


Existing SSAFY merchant names: 16439
Skipped duplicates: 3193
Pending registrations: 202905


Registering merchants:   0%|          | 177/202905 [03:57<75:28:15,  1.34s/merchant, The9900샵]                           


KeyboardInterrupt: 

In [ ]:
print('Skipped duplicate preview:')
pprint(skipped_merchant_names[:10])

print('\nSuccess preview:')
pprint(success_rows[:5])

print('\nFailure preview:')
pprint(failure_rows[:5])
